# 09 - Serious bleeding by bleeding site and by DOAC type

Class-level hazard ratios of serious bleeding for each bleeding site (gastrointestinal, intracranial, urogenital, respiratory, other), and class-by-DOAC interaction models (apixaban reference).

**Manuscript items:** Tables S6, S7; Figs S4, S5

**Inputs:** derived cohort files from 01, comorbidity table from 02, CPRD DrugIssue extract, product dictionary, dosage lookup, interacting-drug code lists; `DERIVED_DIR/major_events_DDI.xlsx`; bleeding code list

**Outputs:** `SUPP_DIR/site_specific_results.xlsx`, `SUPP_DIR/doac_type_and_drug_class_results.xlsx`, `SUPP_FIGURES_DIR/site_specific_HR_plot.png`, `SUPP_FIGURES_DIR/bleeding_HR_by_DOAC_type.png`

*Derived from the original analysis notebook `hr_bleeding_site_and_doac_type.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

In [ ]:
# Cap BLAS/OpenMP threads BEFORE numpy is imported. Without this each worker
# process below spawns its own thread pool and the machine thrashes.
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")

import time
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import multiprocessing as mp
from multiprocessing import Pool
from lifelines import CoxTimeVaryingFitter


In [ ]:
serious_doac_exposure = pd.read_excel(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx"), dtype=str)
serious_doac_exposure["age_at_study_start"] = serious_doac_exposure["age_at_study_start"].astype(int)
serious_doac_exposure["study_start_date"] = pd.to_datetime(serious_doac_exposure["study_start_date"])
serious_doac_exposure["study_end_date"] = pd.to_datetime(serious_doac_exposure["study_end_date"])
serious_doac_exposure["event_date"] = pd.to_datetime(serious_doac_exposure["event_date"])


#doac_exposure["event_date"] = pd.to_datetime(doac_exposure["event_date"])

serious_doac_exposure = serious_doac_exposure[(serious_doac_exposure["age_at_study_start"] >= 18) &
                (serious_doac_exposure["atleast_1y_med_hist_flag"] == "1") &
                (serious_doac_exposure["taken_warfarin_1w_before"] =="0") &
                 (serious_doac_exposure["taken_warfarin_during_study"] =="0") ]

# age_at_study_start is retained here: it is the authoritative source for the
# backfill in the covariate cell below. Dropping it forced age to be re-derived
# from the comorbidity workbook, which is where the NaNs came from.
serious_doac_exposure = serious_doac_exposure[["patid", "study_start_date", "study_end_date", "event_date", "age_at_study_start"]]
print(f"Number of patients in the cohort: {len(serious_doac_exposure)}")
print(f"Number of patients with serious event: {len(serious_doac_exposure[~serious_doac_exposure['event_date'].isna()])}")

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)
#prod_dict.dropna(subset=["DrugSubstanceName"], inplace=True)

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["doac_start_date"] = pats_data["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"])
pats_data["doac_start_date"] = pd.to_datetime(pats_data["doac_start_date"], errors="coerce")
pats_data = pats_data[pats_data["patid"].isin(serious_doac_exposure["patid"])]
pats_data["age"] = (pats_data["doac_start_date"] - pd.to_datetime(pats_data["yob"], format="%Y")).dt.days // 365.25
## add the comorbidities to serious_doac_exposure
#serious_doac_exposure["age"] = serious_doac_exposure["patid"].map(pats_data.set_index("patid")["age"])

In [ ]:
a = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_DrugInteractions_Table.xlsx"), dtype=str)
a = a[["Drug", "Category", "Action", "PK/PD"]]

drugs_action_increase = a[a["Action"] == "Increased"]
drugs_action_decrease = a[a["Action"] == "Decreased"]

In [ ]:
drug_classes_to_consider = ["NSAID", "Antiplatelet", "SSRI", "CCB", "SNRI", "Antiarrythmic"]


In [ ]:
interacting_drugs = pd.read_excel(os.path.join(CODE_LIST_DIR, "DOAC_interacting_drugs.xlsx"), dtype=str)
interacting_drugs = interacting_drugs[interacting_drugs["DrugSubstanceName"].str.contains("/", case=False, na=False) == False]
interacting_drugs["DrugSubstanceName"] = np.where(interacting_drugs["DrugSubstanceName"].isna(), interacting_drugs["matched_drugs"], interacting_drugs["DrugSubstanceName"])

interacting_drugs = interacting_drugs[
    interacting_drugs.apply(
        lambda row: row["DrugSubstanceName"].startswith(row["matched_drugs"][:3])
        if pd.notna(row["DrugSubstanceName"]) else False,
        axis=1
    )
]
words_to_remove = ["eye drops", "inhaler", "cream", "ointment", "gel", "spray", "lotion", "shampoo", "cream"]
interacting_drugs = interacting_drugs[interacting_drugs["Term from EMIS"].str.contains("|".join(words_to_remove), case=False, na=False) == False]
interacting_drugs_codes = interacting_drugs[["ProdCodeId", "matched_drugs"]]
interacting_drugs_codes.rename(columns={"matched_drugs": "drug_name", "ProdCodeId":"prodcodeid"}, inplace=True)
interacting_drugs_codes.drop_duplicates(subset=["prodcodeid"], inplace=True)
interacting_drugs_codes["drug_class"] = interacting_drugs_codes["drug_name"].map(a.set_index("Drug")["Category"])
interacting_drugs_codes = interacting_drugs_codes[interacting_drugs_codes["drug_class"].isin(drug_classes_to_consider)]
interacting_drugs_codes

In [ ]:
ppi_drug_codes= prod_dict[prod_dict["drug_class"].str.contains("Proton Pump Inhibitor", case=False, na=False)]
ppi_drug_codes = ppi_drug_codes[~ppi_drug_codes["DrugSubstanceName"].str.contains("/", case=False, na=False)]
ppi_drug_codes.rename(columns={"DrugSubstanceName":"drug_name"}, inplace=True)
ppi_drug_codes = ppi_drug_codes[["ProdCodeId", "drug_name"]]
ppi_drug_codes["drug_name"] = ppi_drug_codes["drug_name"].replace({
    "Rabeprazole sodium": "Rabeprazole",
    "Pantoprazole sodium sesquihydrate": "Pantoprazole"
})
ppi_vtm_codes= prod_dict[prod_dict["drug_class"].str.contains("Proton Pump Inhibitor", case=False, na=False)]["VTM"].unique().tolist()
ppi_drug_codes = prod_dict[prod_dict["VTM"].isin(ppi_vtm_codes)]
ppi_drug_codes = ppi_drug_codes[~ppi_drug_codes["DrugSubstanceName"].str.contains("/", case=False, na=False)]

ppi_drug_codes.rename(columns={"DrugSubstanceName":"drug_name"}, inplace=True)
ppi_drug_codes  = ppi_drug_codes[["ProdCodeId", "drug_name"]].reset_index(drop=True)
ppi_drug_codes["drug_name"] = ppi_drug_codes["drug_name"].replace({
    "Rabeprazole sodium": "Rabeprazole",
    "Pantoprazole sodium sesquihydrate": "Pantoprazole"
})
ppi_drug_codes.rename(columns={"ProdCodeId":"prodcodeid"}, inplace=True)
ppi_drug_codes["drug_class"] = "PPI"

# interacting_drugs_codes = pd.concat([interacting_drugs_codes, ppi_drug_codes]).reset_index(drop=True)
# interacting_drugs_codes.drop_duplicates(subset=["prodcodeid"], inplace=True)
# interacting_drugs_codes["drug_class"] = interacting_drugs_codes["drug_name"].map(a.set_index("Drug")["Category"])
# interacting_drugs_codes

In [ ]:
import zipfile
dosage = pd.read_csv(os.path.join(LOOKUP_DIR, "common_dosages.txt"), sep="\t",dtype=str)


zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Filtering
    df = df[(df['prodcodeid'].isin(interacting_drugs_codes["prodcodeid"])) & (df['patid'].isin(serious_doac_exposure["patid"].tolist()))]

    df["daily_dose"] = df["dosageid"].map(dosage.set_index("dosageid")["daily_dose"])
    df["study_start_date"] = df["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"])
    df["study_end_date"] = df["patid"].map(serious_doac_exposure.set_index("patid")["study_end_date"])
    
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')
    df["duration"] = pd.to_numeric(df["duration"], errors='coerce')
    df["quantity"] = pd.to_numeric(df["quantity"], errors='coerce')
    df["daily_dose"] = pd.to_numeric(df["daily_dose"], errors='coerce').fillna(0)

    #### Calculate new_duration column

    mask1 = ((df["duration"] == 0) &
            (df["quantity"].notnull()) & 
            (df["daily_dose"] != 0) & 
            (df["quantity"] != 0))

    mask2 = ((df["duration"] == 0) &
            (df["quantity"].notnull()) &
            (df["daily_dose"]==0)
            )


    df["new_duration"] = np.where(mask1, df["quantity"]/df["daily_dose"], df["duration"])  ## apply mask1 
    df["new_duration"] = pd.to_numeric(df["new_duration"], errors="coerce") 

    ## for mask2 get the median quantity for each prodcodeid and fill the new_duration with the median value for that prodcodeid
    median_durations = df[~mask2].groupby("prodcodeid")["duration"].median()
    df["new_duration"] = np.where(mask2, df["prodcodeid"].map(median_durations), df["new_duration"]) ## apply mask2

    df["enddate"] = df["issuedate"] + pd.to_timedelta(df["new_duration"], unit='D')

    df = df[(df["issuedate"] < df["study_end_date"]) &(df["enddate"] > df["study_start_date"])]

    return df


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    # Combine results
    interacting_drugs_exposure = pd.concat(all_results, ignore_index=True)

interacting_drugs_exposure["enddate"] = pd.to_datetime(interacting_drugs_exposure["enddate"], errors='coerce')
interacting_drugs_exposure["enddate"] = interacting_drugs_exposure["enddate"] + pd.to_timedelta(1, unit='D')
interacting_drugs_exposure.drop(columns=["dosageid", "duration", "quantity", "daily_dose", "probobsid", "new_duration"], inplace=True)
interacting_drugs_exposure.sort_values(['patid', 'issuedate'], inplace=True)
interacting_drugs_exposure.reset_index(drop=True, inplace=True)
interacting_drugs_exposure["drug_class"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_class"])
interacting_drugs_exposure["drug_name"] = interacting_drugs_exposure["prodcodeid"].map(interacting_drugs_codes.set_index("prodcodeid")["drug_name"])
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_name"].isin(drugs_action_increase["Drug"].tolist())]
interacting_drugs_exposure = interacting_drugs_exposure[interacting_drugs_exposure["drug_class"].isin(drug_classes_to_consider)]

In [ ]:
major_interacting_drugs_presc = interacting_drugs_exposure.copy()
major_interacting_drugs_presc["study_end_date"] = major_interacting_drugs_presc["patid"].map(serious_doac_exposure.set_index("patid")["study_end_date"])
major_interacting_drugs_presc = major_interacting_drugs_presc[(major_interacting_drugs_presc["issuedate"] < major_interacting_drugs_presc["study_end_date"]) ]
major_interacting_drugs_presc["enddate"] = np.where(major_interacting_drugs_presc["enddate"] > major_interacting_drugs_presc["study_end_date"], 
                                                    major_interacting_drugs_presc["study_end_date"], major_interacting_drugs_presc["enddate"])
major_interacting_drugs_presc.reset_index(drop=True, inplace=True)
major_interacting_drugs_presc["drug_term"] = major_interacting_drugs_presc["prodcodeid"].map(prod_dict.set_index("ProdCodeId")["Term from EMIS"])
major_interacting_drugs_presc

In [ ]:
serious_events_df = serious_doac_exposure[serious_doac_exposure["event_date"].notnull()]

In [ ]:
# Keep only required columns
presc = major_interacting_drugs_presc[["patid", "issuedate", "enddate", "drug_class"]]
events = serious_events_df.reset_index(names="event_row")

merged = events.merge(presc,on="patid",how="left")
active = merged[(merged["issuedate"] <= merged["event_date"]) &(merged["enddate"] >= merged["event_date"])]
active_drugs = (active.groupby("event_row")["drug_class"].agg(list))
serious_events_df["active_drugs_ls"] = (serious_events_df.index.map(active_drugs))
serious_events_df["active_drugs_ls"] = (serious_events_df["active_drugs_ls"].apply(lambda x: x if isinstance(x, list) else []))

serious_events_df

In [ ]:
serious_events = pd.read_excel(os.path.join(DERIVED_DIR, "major_events_DDI.xlsx"), dtype=str)
serious_events["obsdate"] = pd.to_datetime(serious_events["obsdate"], errors='coerce')
serious_events = serious_events[serious_events["patid"].isin(serious_events_df["patid"])]
serious_events

In [ ]:
def process_group(group):
    group = group.sort_values('issuedate').reset_index(drop=True)
    merged = []

    # Set days_gap based on drug_class
    drug_class = group['drug_class'].iloc[0]  # Assuming all rows in the group have the same drug_class
    if drug_class == 'Antibiotic':
        days_gap = 7
    elif drug_class == 'Antifungal':
        days_gap = 7
    else:
        days_gap = 14  # default

    current_start = group.loc[0, 'issuedate']
    current_end = group.loc[0, 'enddate']

    for i in range(1, len(group)):
        next_start = group.loc[i, 'issuedate']
        next_end = group.loc[i, 'enddate']

        if (next_start - current_end).days < days_gap:
            current_end = max(current_end, next_end)
        else:
            merged.append({'patid': group.loc[i-1, 'patid'],
                           'drug_class': group.loc[i-1, 'drug_class'],
                           'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                           'enddate': min(current_end, group['study_end_date'].iloc[0])})
            current_start = next_start
            current_end = next_end

    merged.append({'patid': group.loc[len(group)-1, 'patid'],
                   'drug_class': group.loc[len(group)-1, 'drug_class'],
                   'issuedate': max(current_start, group['study_start_date'].iloc[0]),
                   'enddate': min(current_end, group['study_end_date'].iloc[0])})
    return pd.DataFrame(merged)

def merge_prescriptions(df):
    # Ensure dates are datetime
    for col in ['issuedate', 'enddate', 'study_start_date', 'study_end_date']:
        df[col] = pd.to_datetime(df[col])

    # Split groups
    groups = [group for _, group in df.groupby(['patid', 'drug_class'])]

    # Multiprocessing
    with Pool(50) as pool:
        result_list = pool.map(process_group, groups)

    merged_df = pd.concat(result_list, ignore_index=True)
    return merged_df

In [ ]:
major_interacting_drugs_presc_merged = merge_prescriptions(major_interacting_drugs_presc)
major_interacting_drugs_presc_merged["study_start_date"] = major_interacting_drugs_presc_merged["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"])
major_interacting_drugs_presc_merged["study_end_date"] = major_interacting_drugs_presc_merged["patid"].map(serious_doac_exposure.set_index("patid")["study_end_date"])
#interacting_drugs_presc_merged["event_date"] = interacting_drugs_presc_merged["patid"].map(events_df.set_index("patid")["event_date"])

major_interacting_drugs_presc_merged

In [ ]:
def _process_patient(args):
    patid, group = args
    
    doac_start = group["study_start_date"].iloc[0]
    doac_end = group["study_end_date"].iloc[0]

    # clip exposures to DOAC period
    group = group.copy()
    group["start"] = group["issuedate"].clip(lower=doac_start, upper=doac_end)
    group["stop"] = group["enddate"].clip(lower=doac_start, upper=doac_end)

    # collect cut points
    cut_points = sorted(set([doac_start, doac_end]) | set(group["start"]) | set(group["stop"]))
    intervals = [(cut_points[i], cut_points[i+1]) for i in range(len(cut_points)-1)]

    rows = []
    for start, stop in intervals:
        if start >= stop:
            continue
        active = group[(group["start"] <= start) & (group["stop"] >= stop)]
        active_drugs = sorted(active["drug_class"].unique().tolist())
        rows.append({
            "patid": patid,
            "start_date": start,
            "end_date": stop,
            "concomitant_drugs_ls": active_drugs
        })
    return rows

def build_exposure_timeline(df, n_jobs=None):
    df = df.copy()
    df['issuedate'] = pd.to_datetime(df['issuedate'])
    df['enddate'] = pd.to_datetime(df['enddate'])
    df['study_start_date'] = pd.to_datetime(df['study_start_date'])
    df['study_end_date'] = pd.to_datetime(df['study_end_date'])
    
    grouped = list(df.groupby("patid"))

    with mp.Pool(processes=60) as pool:
        results = pool.map(_process_patient, grouped)

    # flatten list of lists
    flat_results = [row for sublist in results for row in sublist]
    return pd.DataFrame(flat_results)

In [ ]:
major_exposure_timeline_df = build_exposure_timeline(major_interacting_drugs_presc_merged)

major_exposure_timeline_df["study_start_date"] = major_exposure_timeline_df["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"])
major_exposure_timeline_df["study_end_date"] = major_exposure_timeline_df["patid"].map(serious_doac_exposure.set_index("patid")["study_end_date"])

major_exposure_timeline_df["event_date"] = major_exposure_timeline_df["patid"].map(serious_events_df.set_index("patid")["event_date"])
major_exposure_timeline_df["event_flag"] = np.where(
    (major_exposure_timeline_df["event_date"] >= major_exposure_timeline_df["start_date"]) & 
    (major_exposure_timeline_df["event_date"] <= major_exposure_timeline_df["end_date"]), 1, 0)

In [ ]:
pats_with_exposures = set(major_exposure_timeline_df["patid"].unique())

unique_serious_events_df = serious_events_df.drop_duplicates(subset=["patid"]).copy()

pats_without_exposures_df = serious_doac_exposure[~serious_doac_exposure["patid"].isin(pats_with_exposures)].copy()
pats_without_exposures_df["concomitant_drugs_ls"] = [[] for _ in range(len(pats_without_exposures_df))]
pats_without_exposures_df["start_date"] = pats_without_exposures_df["study_start_date"]
pats_without_exposures_df["end_date"] = pats_without_exposures_df["study_end_date"]
pats_without_exposures_df["event_date"] = pats_without_exposures_df["patid"].map(
    unique_serious_events_df.set_index("patid")["event_date"]
)
pats_without_exposures_df["event_flag"] = np.where(
    (pats_without_exposures_df["event_date"] >= pats_without_exposures_df["start_date"]) &
    (pats_without_exposures_df["event_date"] <= pats_without_exposures_df["end_date"]),
    1, 0
)

major_exposure_timeline_df = pd.concat([major_exposure_timeline_df, pats_without_exposures_df], ignore_index=True)
major_exposure_timeline_df.reset_index(drop=True, inplace=True)

In [ ]:
major_df = major_exposure_timeline_df.copy()
major_df = major_df[major_df["start_date"] < major_df["end_date"]].reset_index(drop=True)

for drug in drug_classes_to_consider:
    major_df[f"{drug}"] = major_df["concomitant_drugs_ls"].apply(lambda drugs: 1 if drug in drugs else 0)


In [ ]:
zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Ensure datetime conversion early
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors='coerce')

    # Filter rows first (KEEP dataframe)
    df = df[
        df['prodcodeid'].isin(ppi_drug_codes["prodcodeid"]) &
        df['patid'].isin(serious_doac_exposure["patid"])
    ].copy()

    # Map study start date
    df["study_start_date"] = df["patid"].map(
        serious_doac_exposure.set_index("patid")["study_start_date"]
    )

    df["study_start_date"] = pd.to_datetime(df["study_start_date"], errors='coerce')

    # Apply time window filter
    df = df[
        (df["issuedate"] >= df["study_start_date"] - pd.Timedelta(days=180)) &
        (df["issuedate"] < df["study_start_date"])
    ]

    return df["patid"].unique().tolist()


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    ppi_pats = set(p for sublist in all_results for p in sublist)

In [ ]:
# Baseline PPI use: a PPI prescription in the 180 days (six months) before the index date.
major_df["PPI_baseline"] = major_df["patid"].isin(ppi_pats).astype(int)
pats_data["gender"] = pats_data["gender"].map({"1": 1, "2": 0})

In [ ]:
# ===========================================================================
# Baseline covariates - RAW values, missing continuous values left as NaN
# ===========================================================================
# This cell no longer reads a pre-imputed workbook. It loads the raw comorbidity
# table so the missing values are still present and can be handled by the
# multiple imputation below. Imputation has to happen after the analysis frames
# exist, because the imputation model needs the outcome and the Nelson-Aalen
# cumulative hazard (White & Royston 2009).

comorbidities = pd.read_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), dtype={"patid": str})

binary_comorbidity_cols = ["major_bled_hist", "hypertension", "liver_disease", "stroke", "gender", "heart_failure",
                           "atrial_fibrillation", "venous_thromboembolism", "cancer", "diabetes", "smoking", "alcohol"]
continuous_comorbidity_cols = ["age_at_study_start", "ckd_value", "bmi_value"]

comorbidities["gender"] = comorbidities["patid"].map(pats_data.set_index("patid")["gender"])

for col in continuous_comorbidity_cols:
    if col in comorbidities.columns:
        comorbidities[col] = pd.to_numeric(comorbidities[col], errors="coerce")


def backfill_age(df, cohort_df, label=""):
    """Age is derivable exactly from year of birth and the index date, so it is
    merged in rather than imputed. Imputing a variable you can compute would not
    survive review."""
    before = int(df["age_at_study_start"].isna().sum()) if "age_at_study_start" in df else len(df)
    age = pd.to_numeric(df.get("age_at_study_start"), errors="coerce")
    if "age_at_study_start" in cohort_df.columns:
        lut = cohort_df.drop_duplicates("patid").set_index("patid")["age_at_study_start"]
        age = age.fillna(df["patid"].map(pd.to_numeric(lut, errors="coerce")))
    if age.isna().any() and "age" in pats_data.columns:
        lut2 = pats_data.drop_duplicates("patid").set_index("patid")["age"]
        age = age.fillna(df["patid"].map(pd.to_numeric(lut2, errors="coerce")))
    df["age_at_study_start"] = age
    after = int(age.isna().sum())
    print(f"[{label}] age missing rows: {before:,} -> {after:,}"
          + ("  <-- still unresolved, investigate before imputing" if after else ""))
    return df


def report_missingness(df, label=""):
    """Patient-level missingness. Cross-check against the manuscript (eGFR 3.7%,
    BMI 8.4%): a mismatch points at the merge, not at the data."""
    pat = df.drop_duplicates("patid")
    rep = pd.DataFrame([
        {"variable": c, "n_missing": int(pat[c].isna().sum()),
         "pct_missing": round(100 * pat[c].isna().mean(), 2)}
        for c in continuous_comorbidity_cols])
    print(f"[{label}] {pat['patid'].nunique():,} patients")
    print(rep.to_string(index=False))
    return rep


_absent = set(major_df["patid"]) - set(comorbidities["patid"])
print(f"Patients missing from the comorbidity workbook entirely: {len(_absent):,} "
      f"({100 * len(_absent) / major_df['patid'].nunique():.2f}%)")

for col in binary_comorbidity_cols:
    major_df[col] = major_df["patid"].map(comorbidities.set_index("patid")[col]).fillna(0).astype(int)

# NaNs deliberately carried through - the imputation step fills them, and the
# complete-case sensitivity analysis needs to know which were originally missing.
for col in continuous_comorbidity_cols:
    major_df[col] = major_df["patid"].map(comorbidities.set_index("patid")[col])

major_df = backfill_age(major_df, serious_doac_exposure, "Serious bleeding")
missing_report_major = report_missingness(major_df, "Serious bleeding")


In [ ]:
# ===========================================================================
# PPI adjustment applied ONLY to the NSAID class
# ===========================================================================
# A single covariate in one joint model necessarily adjusts every class's hazard
# ratio, so each model below is fitted twice - once with PPI_baseline and once
# without - and each reported row is taken from the appropriate fit. A
# "PPI adjusted" column records the provenance. State this in the methods: the
# published tables mix estimates from two models.
#
# Only NSAID is adjusted. Aspirin has no term of its own here; it sits inside
# the Antiplatelet class alongside clopidogrel, which is left unadjusted.

PPI_ADJUSTED_CLASSES = ["NSAID"]

_missing = [c for c in PPI_ADJUSTED_CLASSES if c not in drug_classes_to_consider]
if _missing:
    raise ValueError(f"PPI_ADJUSTED_CLASSES not in drug_classes_to_consider: {_missing}")
print("Classes adjusted for baseline PPI use:", PPI_ADJUSTED_CLASSES)

comorbidities_covariates = " + ".join(binary_comorbidity_cols + continuous_comorbidity_cols)

# ---- Site-specific formulas ----------------------------------------------
covariates_no_ppi = " + ".join(drug_classes_to_consider) + " + " + comorbidities_covariates
covariates_with_ppi = covariates_no_ppi + " + PPI_baseline"

# Kept so any cell still referencing `covariates` gets the unadjusted formula.
covariates = covariates_no_ppi


In [ ]:
# ---------- Parallel Cox fitting ----------------------------------------------
# The five site-specific models are independent of each other, and each is now
# fitted twice (with and without PPI_baseline), so there are ten independent
# fits. They all go into one pool. The data frame is published to a module-level
# global BEFORE the Pool is created: on Linux the workers are forked and inherit
# it copy-on-write, so a large frame is never pickled or duplicated. Same pattern
# the DrugIssue readers above already use. Requires the default 'fork' start
# method (mp.get_start_method() should return 'fork').

FIT_N_JOBS = 12

_FIT_DF = None   # inherited by forked workers, never pickled


def _fit_worker(task):
    key, formula, cols, event_col = task
    ctv = CoxTimeVaryingFitter()
    ctv.fit(
        _FIT_DF[cols],
        id_col='patid', start_col='start_date', stop_col='end_date',
        event_col=event_col, formula=formula, show_progress=False,
    )
    # lifelines labels the covariance columns positionally in some versions;
    # relabel both axes so downstream .at[] lookups work by name.
    vcov = pd.DataFrame(np.asarray(ctv.variance_matrix_),
                        index=ctv.params_.index, columns=ctv.params_.index)
    return key, {"summary": ctv.summary, "params": ctv.params_, "vcov": vcov}


def fit_models(df, specs, label="", n_jobs=FIT_N_JOBS):
    """specs: {key: (formula, cols, event_col)}. Returns {key: {summary, params, vcov}}.

    Fitted models are not returned - shipping one back from a worker is
    expensive - but summary/params/vcov cover everything used downstream.
    """
    global _FIT_DF
    tasks = [(k, f, c, e) for k, (f, c, e) in specs.items()]
    t0 = time.time()
    _FIT_DF = df
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            results = pool.map(_fit_worker, tasks)
    finally:
        _FIT_DF = None
    print(f"[{label}] fitted {len(results)} models in {time.time()-t0:.1f}s")
    return dict(results)


def model_for(fits, drug, prefix=""):
    """Pick the PPI-adjusted or unadjusted fit for a given drug class."""
    suffix = "with_ppi" if drug in PPI_ADJUSTED_CLASSES else "no_ppi"
    return fits[f"{prefix}{suffix}"]


def ppi_flag(drug):
    return "Yes" if drug in PPI_ADJUSTED_CLASSES else "No"


## Multiple imputation of missing continuous covariates

Missing values in the continuous baseline covariates (`ckd_value` / eGFR and
`bmi_value`) are handled by **multiple imputation by chained equations (MICE)**
using `miceforest` (random forest / LightGBM chained equations with predictive
mean matching), rather than by complete-case analysis or a missing-indicator
category.

Three things make this survival-analysis-appropriate rather than a generic MICE run:

1. **The outcome is in the imputation model.** The event indicator, follow-up
   time and the **Nelson-Aalen cumulative hazard** estimate are included as
   predictors, following White IR & Royston P, *Stat Med* 2009;28:1982-1998.
   Omitting the outcome biases covariate-outcome associations toward the null.
2. **The imputation model is congenial with the substantive model.** Every
   covariate in the Cox models is a predictor in the imputation model, plus
   ever-exposed indicators for each interacting drug and baseline PPI use, so
   the imputation model is at least as rich as the analysis model.
3. **Estimates are pooled by Rubin's rules** on the log-hazard scale, with
   Barnard-Rubin degrees of freedom, and the fraction of missing information
   (FMI) is reported for each coefficient.

Imputation is run **separately for each outcome cohort** (serious, minor, all
bleeding), because the outcome and follow-up time entering the imputation model
differ between them.

A **complete-case sensitivity analysis** is run alongside, as recommended by
Sterne JAC et al., *BMJ* 2009;338:b2393 and required for STROBE / RECORD-PE
reporting.


In [ ]:
# ===========================================================================
# Multiple imputation helpers (MICE + Rubin's rules)
# ===========================================================================
from scipy import stats
from lifelines import NelsonAalenFitter

# ---- Configuration --------------------------------------------------------
# m = number of imputed datasets. Set to 10 to match the manuscript and the
# White/Royston/Wood (2011) rule of thumb that m should be at least the
# percentage of incomplete cases (BMI ~8.4% missing). Set to 5 if you prefer
# the smaller number - nothing else needs to change.
MI_CONFIG = dict(
    m=10,
    iterations=5,
    seed=20240824,
    mean_match_candidates=5,   # predictive mean matching: imputed values are
                               # drawn from observed values of near neighbours
    n_jobs=10,                 # imputed datasets are independent draws, so each
                               # one is built in its own worker process. Set to 1
                               # to fall back to a single m-dataset kernel.
)
IMPUTE_COLS = ["ckd_value", "bmi_value"]   # continuous covariates to impute

# The 2 x m adjusted fits per outcome are all independent, so they are farmed
# out as a single flat pool of tasks rather than looping over imputations. Each
# worker builds its own copy of the design frame, so peak memory is roughly
# MI_FIT_N_JOBS x (size of the interval-level frame restricted to model
# columns). Lower this first if the node runs out of memory.
MI_FIT_N_JOBS = 20

# NOTE ON THREADS: the first cell of this notebook pins OMP_NUM_THREADS=1, which
# also silences LightGBM's internal threading inside miceforest. That is the
# right default here because MICE is parallelised across processes instead.
# If you set MI_CONFIG["n_jobs"]=1, give LightGBM threads back explicitly:
#     run_mice(..., num_threads=16)      # **lgb_kwargs are forwarded to mice()
# Do not do both at once - the two layers will oversubscribe the cores.


def check_mi_versions():
    """miceforest 6.0.x is incompatible with lightgbm >= 4.6 (the
    Booster.__inner_predict signature changed). Fail loudly and early rather
    than 40 minutes into a run."""
    import miceforest, lightgbm
    print(f"miceforest {miceforest.__version__} | lightgbm {lightgbm.__version__}")
    major_minor = tuple(int(x) for x in lightgbm.__version__.split(".")[:2])
    if major_minor >= (4, 6):
        print("WARNING: kernel.mice() may raise "
              "'Booster.__inner_predict() takes 1 positional argument'. "
              "If it does, run:  pip install 'lightgbm<4.6'")


# ---- 1. Nelson-Aalen cumulative hazard (White & Royston 2009) --------------
def nelson_aalen_cumhaz(durations, events):
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)
    naf = NelsonAalenFitter()
    naf.fit(durations, event_observed=events)
    ch = naf.cumulative_hazard_.iloc[:, 0]
    times = ch.index.to_numpy(dtype=float)
    idx = np.clip(np.searchsorted(times, durations, side="right") - 1, 0, len(times) - 1)
    return ch.to_numpy()[idx]


# ---- 2. Patient-level frame for the imputation model ----------------------
def build_imputation_frame(analysis_df, comorbidities, binary_cols, continuous_cols,
                           drug_terms=(), extra_binary=("PPI_baseline",),
                           event_col="event_flag"):
    """One row per patient: total follow-up, event indicator, Nelson-Aalen
    cumulative hazard, every analysis covariate, and ever-exposed flags for the
    interacting drugs (so the imputation model is congenial with the Cox model)."""
    df = analysis_df
    terms = [t for t in list(drug_terms) + list(extra_binary) if t in df.columns]

    slim = pd.DataFrame({
        "patid": df["patid"].to_numpy(),
        "_dur": (df["end_date"] - df["start_date"]).dt.days.clip(lower=0).to_numpy(),
        "event_flag": df[event_col].to_numpy(),
    })
    for t in terms:
        slim[t] = df[t].to_numpy().astype("int8")

    agg_map = {"fu_days": ("_dur", "sum"), "event": ("event_flag", "max")}
    agg_map.update({f"ever_{t}": (t, "max") for t in terms})
    pat = slim.groupby("patid", sort=False).agg(**agg_map).reset_index()

    pat["fu_years"] = (pat.pop("fu_days") / 365.25).clip(lower=1 / 365.25)
    pat["event"] = pat["event"].astype(int)
    pat["na_cumhaz"] = nelson_aalen_cumhaz(pat["fu_years"], pat["event"])
    pat["log_fu_years"] = np.log(pat["fu_years"])
    for t in terms:
        pat[f"ever_{t}"] = pat[f"ever_{t}"].astype(float)

    # Covariate values are taken from the ANALYSIS frame, not from the workbook.
    # That frame is what the Cox model actually sees, and it carries any
    # correction applied after the merge - notably the age backfill. Reading from
    # the workbook here would silently undo those corrections and hand MICE a
    # column it was never meant to impute. The workbook is only a fallback for
    # columns the analysis frame does not carry.
    first = analysis_df.drop_duplicates("patid").set_index("patid")
    com = (comorbidities.drop_duplicates(subset=["patid"]).set_index("patid")
           if comorbidities is not None else None)

    def _pull(col):
        if col in first.columns:
            return pd.to_numeric(pat["patid"].map(first[col]), errors="coerce")
        if com is not None and col in com.columns:
            return pd.to_numeric(pat["patid"].map(com[col]), errors="coerce")
        return None

    for col in binary_cols:
        v = _pull(col)
        if v is not None:
            pat[col] = v.fillna(0).astype(float)
    for col in continuous_cols:
        v = _pull(col)
        if v is not None:
            pat[col] = v

    return pat


# ---- 3. MICE with miceforest ----------------------------------------------
def _make_kernel(work, m, seed, mean_match_candidates=5):
    """Construct an ImputationKernel across miceforest 5.x / 6.x signatures."""
    import miceforest as mf
    attempts = [
        dict(num_datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations_data=True),
        dict(datasets=m, random_state=seed, mean_match_candidates=mean_match_candidates,
             save_all_iterations=True),
        dict(num_datasets=m, random_state=seed),
        dict(datasets=m, random_state=seed),
    ]
    last = None
    for kw in attempts:
        try:
            return mf.ImputationKernel(work, **kw)
        except TypeError as exc:
            last = exc
    raise RuntimeError(f"Could not construct miceforest ImputationKernel: {last}")


_MICE_WORK = None       # published to forked workers, never pickled
_MICE_TARGETS = None


def _mice_worker(args):
    """One imputed dataset, built by its own single-dataset kernel."""
    idx, seed, iterations, mmc, lgb_kwargs = args
    kernel = _make_kernel(_MICE_WORK, 1, seed, mmc)
    kernel.mice(iterations, **lgb_kwargs)
    comp = kernel.complete_data(dataset=0)
    return idx, {c: np.asarray(comp[c], dtype=float) for c in _MICE_TARGETS}


def run_mice_parallel(pat_frame, continuous_cols, targets, m, iterations, seed,
                      mean_match_candidates, n_jobs, label="", **lgb_kwargs):
    """m independent chains in m worker processes.

    Statistically this is the same estimator as one m-dataset kernel: the m
    imputations are independent draws either way, the only difference being that
    each chain gets its own RNG stream. Streams are spawned from a single
    SeedSequence so the whole run is reproducible from MI_CONFIG["seed"]."""
    global _MICE_WORK, _MICE_TARGETS
    t0 = time.time()
    ss = np.random.SeedSequence(seed)
    seeds = [int(s.generate_state(1)[0]) % (2 ** 31 - 1) for s in ss.spawn(m)]
    tasks = [(i, seeds[i], iterations, mean_match_candidates, lgb_kwargs) for i in range(m)]

    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    _MICE_WORK, _MICE_TARGETS = work, targets
    try:
        with Pool(min(n_jobs, m)) as pool:
            results = dict(pool.map(_mice_worker, tasks))
    finally:
        _MICE_WORK = _MICE_TARGETS = None

    out = []
    for i in range(m):
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in results[i]:
                frame[c] = results[i][c]
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE: {m} datasets on {min(n_jobs, m)} workers "
          f"in {time.time() - t0:.1f}s | seeds {seeds[:3]}...")
    return out, seeds


def run_mice(pat_frame, continuous_cols, m=10, iterations=5, seed=20240824,
             mean_match_candidates=5, n_jobs=1, label="", verbose=True, **lgb_kwargs):
    """Returns (list of m patid->imputed-value frames, fitted kernel or seeds)."""
    t0 = time.time()
    work = pat_frame.drop(columns=["patid"]).copy()
    work = work.apply(pd.to_numeric, errors="coerce").astype("float64")

    target_cols = [c for c in work.columns if work[c].isna().any()]

    # A column with no observed values at all is a merge failure, not missing
    # data. miceforest would otherwise die inside a numpy sampling call with an
    # error that says nothing about which column is at fault.
    _empty = [c for c in target_cols if int(work[c].notna().sum()) == 0]
    if _empty:
        raise ValueError(
            f"These columns are 100% missing, so there is nothing to impute from: "
            f"{_empty}. Fix the source merge before imputing.")
    _sparse = {c: round(100 * work[c].notna().mean(), 2) for c in target_cols
               if work[c].notna().mean() < 0.5}
    if _sparse:
        print(f"[{label}] WARNING: under half observed for {_sparse} (% observed) - "
              "check the merge before trusting these imputations.")

    if verbose:
        print(f"[{label}] imputing {target_cols} | m={m}, iterations={iterations}, "
              f"predictors={work.shape[1] - 1}, n={len(work):,}")
        extra = [c for c in target_cols if c not in continuous_cols]
        if extra:
            print(f"[{label}] note: these predictors also had missing values and "
                  f"were imputed jointly: {extra}")
    if not target_cols:
        print(f"[{label}] nothing to impute - no missing values found.")
        base = pat_frame[["patid"] + [c for c in continuous_cols if c in pat_frame.columns]].copy()
        return [base.copy() for _ in range(m)], None

    if n_jobs and n_jobs > 1 and m > 1:
        return run_mice_parallel(pat_frame, continuous_cols, target_cols, m, iterations,
                                 seed, mean_match_candidates, n_jobs, label=label,
                                 **lgb_kwargs)

    kernel = _make_kernel(work, m, seed, mean_match_candidates)
    kernel.mice(iterations, **lgb_kwargs)

    out = []
    for i in range(m):
        comp = kernel.complete_data(dataset=i)
        frame = pd.DataFrame({"patid": pat_frame["patid"].to_numpy()})
        for c in continuous_cols:
            if c in comp.columns:
                frame[c] = np.asarray(comp[c])
            elif c in pat_frame.columns:
                frame[c] = pat_frame[c].to_numpy()
        out.append(frame)
    print(f"[{label}] MICE finished in {time.time() - t0:.1f}s")
    return out, kernel


def imputation_diagnostics(pat_frame, imputations, continuous_cols):
    """Observed vs imputed distributions - include in the supplement."""
    rows = []
    for c in continuous_cols:
        if c not in pat_frame.columns:
            continue
        miss = pat_frame[c].isna().to_numpy()
        obs = pat_frame.loc[~miss, c]
        rows.append(dict(variable=c, source="observed", n=int(obs.size), mean=obs.mean(),
                         sd=obs.std(), p25=obs.quantile(.25), median=obs.median(),
                         p75=obs.quantile(.75)))
        if miss.sum() == 0:
            continue
        for i, imp in enumerate(imputations, 1):
            v = pd.to_numeric(imp.loc[miss, c])
            rows.append(dict(variable=c, source=f"imputed {i}", n=int(v.size), mean=v.mean(),
                             sd=v.std(), p25=v.quantile(.25), median=v.median(),
                             p75=v.quantile(.75)))
    return pd.DataFrame(rows).round(3)


# ---- 4. Rubin's rules ------------------------------------------------------
def rubin_pool(summaries, n_events=None, n_params=None):
    """Pool lifelines `.summary` frames on the log-hazard scale.

    Returns a frame shaped like a lifelines summary (so it can be passed
    straight to build_bleeding_summary) plus FMI, RIV and Barnard-Rubin df."""
    coef = pd.concat([s["coef"] for s in summaries], axis=1)
    se = pd.concat([s["se(coef)"] for s in summaries], axis=1)
    m = coef.shape[1]
    if m < 2:
        raise ValueError("Rubin pooling needs at least 2 imputed datasets")

    qbar = coef.mean(axis=1)                 # pooled point estimate
    ubar = (se ** 2).mean(axis=1)            # within-imputation variance
    b = coef.var(axis=1, ddof=1)             # between-imputation variance
    total = ubar + (1 + 1 / m) * b           # total variance
    se_pool = np.sqrt(total)

    lam = np.where(total > 0, ((1 + 1 / m) * b) / total, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(ubar > 0, ((1 + 1 / m) * b) / ubar, 0.0)
        df_old = np.where(r > 0, (m - 1) * (1 + 1 / r) ** 2, np.inf)

    if n_events and n_params:                # Barnard & Rubin (1999) adjustment
        nu_com = max(int(n_events) - int(n_params), 1)
        df_obs = ((nu_com + 1) / (nu_com + 3)) * nu_com * (1 - lam)
        df_adj = np.where(np.isfinite(df_old), 1 / (1 / df_old + 1 / df_obs), float(nu_com))
    else:
        df_adj = np.where(np.isfinite(df_old), df_old, 1e6)
    df_adj = np.clip(df_adj, 1.0, 1e7)

    crit = stats.t.ppf(0.975, df_adj)
    zstat = qbar / se_pool
    out = pd.DataFrame({
        "coef": qbar,
        "se(coef)": se_pool,
        "exp(coef)": np.exp(qbar),
        "exp(coef) lower 95%": np.exp(qbar - crit * se_pool),
        "exp(coef) upper 95%": np.exp(qbar + crit * se_pool),
        "coef lower 95%": qbar - crit * se_pool,
        "coef upper 95%": qbar + crit * se_pool,
        "z": zstat,
        "p": 2 * stats.t.sf(np.abs(zstat), df_adj),
        "fmi": lam + 2 / (df_adj + 3),
        "riv": r,
        "df": df_adj,
        "n_imputations": m,
    })
    out.index.name = "covariate"
    return out


# ---- 5. Drivers: fit across imputations, pool, and complete-case fallback --


# ---------------------------------------------------------------------------
# 6. Multivariate Rubin's rules (pooled parameter vector AND covariance matrix)
# ---------------------------------------------------------------------------
# The scalar pooling above is enough when every reported number is a single
# coefficient. It is NOT enough here: some tables below combine coefficients
# (a main effect plus its interaction term) and need the covariance between
# them. Rubin's rules extend to the whole vector - the total variance matrix is
# T = Ubar + (1 + 1/m) B, where Ubar is the average within-imputation covariance
# matrix and B is the between-imputation covariance of the parameter vectors.
# Pooling only the diagonal would understate the covariance and give wrong
# confidence intervals for any derived contrast.

def rubin_pool_mv(params_list, vcov_list, n_events=None):
    """Pool a parameter vector and its covariance matrix across imputations.

    Returns {"summary", "params", "vcov"} so it is a drop-in replacement for a
    single fit's results."""
    P = pd.concat([pd.Series(p) for p in params_list], axis=1)
    m = P.shape[1]
    if m < 2:
        raise ValueError("Rubin pooling needs at least 2 imputed datasets")
    names = P.index

    qbar = P.mean(axis=1)
    Ubar = sum(v.reindex(index=names, columns=names) for v in vcov_list) / m
    dev = P.sub(qbar, axis=0).to_numpy()
    B = pd.DataFrame(dev @ dev.T / (m - 1), index=names, columns=names)
    T = Ubar + (1 + 1 / m) * B

    u_diag = pd.Series(np.diag(Ubar.to_numpy()), index=names)
    b_diag = pd.Series(np.diag(B.to_numpy()), index=names)
    t_diag = pd.Series(np.diag(T.to_numpy()), index=names)
    se_pool = np.sqrt(t_diag)

    lam = np.where(t_diag > 0, ((1 + 1 / m) * b_diag) / t_diag, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(u_diag > 0, ((1 + 1 / m) * b_diag) / u_diag, 0.0)
        df_old = np.where(r > 0, (m - 1) * (1 + 1 / r) ** 2, np.inf)
    if n_events:
        nu_com = max(int(n_events) - len(names), 1)
        df_obs = ((nu_com + 1) / (nu_com + 3)) * nu_com * (1 - lam)
        df_adj = np.where(np.isfinite(df_old), 1 / (1 / df_old + 1 / df_obs), float(nu_com))
    else:
        df_adj = np.where(np.isfinite(df_old), df_old, 1e6)
    df_adj = np.clip(df_adj, 1.0, 1e7)

    crit = stats.t.ppf(0.975, df_adj)
    zstat = qbar / se_pool
    summary = pd.DataFrame({
        "coef": qbar,
        "se(coef)": se_pool,
        "exp(coef)": np.exp(qbar),
        "exp(coef) lower 95%": np.exp(qbar - crit * se_pool),
        "exp(coef) upper 95%": np.exp(qbar + crit * se_pool),
        "z": zstat,
        "p": 2 * stats.t.sf(np.abs(zstat), df_adj),
        "fmi": lam + 2 / (df_adj + 3),
        "df": df_adj,
        "n_imputations": m,
    })
    summary.index.name = "covariate"
    return {"summary": summary, "params": qbar, "vcov": T}


In [ ]:
# ---- Adapter: MI driver matching this notebook's fit_models() ---------------
# fit_models() takes {key: (formula, cols, event_col)} and returns
# {key: {summary, params, vcov}}. The MI version runs every spec on every imputed
# dataset and pools with the MULTIVARIATE Rubin rules, because the DOAC-type
# table below combines a main effect with its interaction term and therefore
# needs the pooled covariance between them, not just the pooled standard errors.

IMPUTE_COLS = ["ckd_value", "bmi_value"]
MI_FIT_N_JOBS = 12

_MI_BASE_DF = None      # inherited copy-on-write by forked workers
_MI_VALUES = None       # per-imputation, row-aligned imputed values


def _fit_mi_worker(args):
    imp_idx, key, formula, cols, event_col = args
    sub = _MI_BASE_DF[cols].copy()
    for c, arr in _MI_VALUES[imp_idx].items():
        if c in sub.columns:
            sub[c] = arr
    ctv = CoxTimeVaryingFitter()
    ctv.fit(sub, id_col="patid", start_col="start_date", stop_col="end_date",
            event_col=event_col, formula=formula, show_progress=False)
    vcov = pd.DataFrame(np.asarray(ctv.variance_matrix_),
                        index=ctv.params_.index, columns=ctv.params_.index)
    return imp_idx, key, ctv.summary, ctv.params_, vcov


def fit_models_mi(df, imputations, specs, impute_cols=None, n_jobs=MI_FIT_N_JOBS,
                  label="", event_col_for_df=None):
    """Same contract as fit_models(), but pooled across imputed datasets."""
    global _MI_BASE_DF, _MI_VALUES
    impute_cols = IMPUTE_COLS if impute_cols is None else list(impute_cols)
    t0 = time.time()

    values = []
    for i, imp in enumerate(imputations, 1):
        lut = imp.set_index("patid")
        vals = {c: df["patid"].map(lut[c]).to_numpy(dtype=float)
                for c in impute_cols if c in lut.columns}
        for c, arr in vals.items():
            n_bad = int(np.isnan(arr).sum())
            if n_bad:
                raise ValueError(f"{c}: {n_bad} rows still NaN after imputation {i} - "
                                 "some patids are absent from the imputation frame.")
        values.append(vals)

    tasks = [(i, k, f, c, e) for i in range(len(imputations))
             for k, (f, c, e) in specs.items()]
    _MI_BASE_DF, _MI_VALUES = df, values
    try:
        with Pool(min(n_jobs, len(tasks))) as pool:
            res = pool.map(_fit_mi_worker, tasks)
    finally:
        _MI_BASE_DF = _MI_VALUES = None

    per_key = {k: {"params": [None] * len(imputations),
                   "vcov": [None] * len(imputations)} for k in specs}
    for imp_idx, key, summ, params, vcov in res:
        per_key[key]["params"][imp_idx] = params
        per_key[key]["vcov"][imp_idx] = vcov

    out = {}
    for key, (_, _, e_col) in specs.items():
        n_ev = int(df.loc[df[e_col] == 1, "patid"].nunique())
        out[key] = rubin_pool_mv(per_key[key]["params"], per_key[key]["vcov"],
                                 n_events=n_ev)
    print(f"[{label}] {len(tasks)} fits on {min(n_jobs, len(tasks))} workers, "
          f"pooled in {time.time() - t0:.1f}s")
    return out


def complete_case_frame(df, impute_cols=None, label=""):
    impute_cols = IMPUTE_COLS if impute_cols is None else impute_cols
    mask = df[list(impute_cols)].notna().all(axis=1)
    sub = df.loc[mask].reset_index(drop=True)
    print(f"[{label}] complete-case: {sub['patid'].nunique():,} of "
          f"{df['patid'].nunique():,} patients "
          f"({100 * sub['patid'].nunique() / df['patid'].nunique():.1f}%)")
    return sub


def compare_mi_vs_cc(mi_df, cc_df, on):
    """Side-by-side MI vs complete-case HRs with the percent change in the point
    estimate - the table reviewers ask for."""
    keys = list(on)
    a = mi_df[keys + ["HR", "CI Lower", "CI Upper", "p"]].rename(
        columns={"HR": "HR (MI)", "CI Lower": "CI Lower (MI)",
                 "CI Upper": "CI Upper (MI)", "p": "p (MI)"})
    b = cc_df[keys + ["HR", "CI Lower", "CI Upper", "p"]].rename(
        columns={"HR": "HR (CC)", "CI Lower": "CI Lower (CC)",
                 "CI Upper": "CI Upper (CC)", "p": "p (CC)"})
    out = a.merge(b, on=keys, how="outer")
    out["% change in HR"] = (100 * (out["HR (CC)"] - out["HR (MI)"]) / out["HR (MI)"]).round(1)
    return out.reset_index(drop=True)


In [ ]:
bleeding_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "bleeding_codes.xlsx"), dtype=str)
serious_events["bleeding_type"] = serious_events["medcodeid"].map(bleeding_codes.set_index("MedCodeId")["Bleeding Category"])
serious_events

In [ ]:
site_major_df = major_df.copy()

gi_bleeding_events_df = serious_events[serious_events["bleeding_type"]=="Gastrointestinal Bleeding"].copy()
intracranial_bleeding_events_df = serious_events[serious_events["bleeding_type"]=="Intracranial Bleeding"].copy()
respiratory_bleeding_events_df = serious_events[serious_events["bleeding_type"]=="Respiratory Bleeding"].copy()
urogenital_bleeding_events_df = serious_events[serious_events["bleeding_type"]=="Urogenital Bleeding"].copy()
other_bleeding_events_df = serious_events[serious_events["bleeding_type"].isin(["Other Bleeding"])].copy()

In [ ]:
site_major_df["gi_bleeding"] = site_major_df[site_major_df["event_flag"]==1]["patid"].isin(gi_bleeding_events_df["patid"]).astype(int)
site_major_df["intracranial_bleeding"] = site_major_df[site_major_df["event_flag"]==1]["patid"].isin(intracranial_bleeding_events_df["patid"]).astype(int)
site_major_df["respiratory_bleeding"] = site_major_df[site_major_df["event_flag"]==1]["patid"].isin(respiratory_bleeding_events_df["patid"]).astype(int)
site_major_df["urogenital_bleeding"] = site_major_df[site_major_df["event_flag"]==1]["patid"].isin(urogenital_bleeding_events_df["patid"]).astype(int)
site_major_df["other_bleeding"] = site_major_df[site_major_df["event_flag"]==1]["patid"].isin(other_bleeding_events_df["patid"]).astype(int)

##fill na with 0 for bleeding types
site_major_df["gi_bleeding"] = site_major_df["gi_bleeding"].fillna(0).astype(int)
site_major_df["intracranial_bleeding"] = site_major_df["intracranial_bleeding"].fillna(0).astype(int)
site_major_df["respiratory_bleeding"] = site_major_df["respiratory_bleeding"].fillna(0).astype(int)
site_major_df["urogenital_bleeding"] = site_major_df["urogenital_bleeding"].fillna(0).astype(int)
site_major_df["other_bleeding"] = site_major_df["other_bleeding"].fillna(0).astype(int)

In [ ]:
bleedintg_sites = ["gi_bleeding", "intracranial_bleeding", "respiratory_bleeding", "urogenital_bleeding", "other_bleeding"]

bleeding_types_and_drug_classes_counts = {}
for drug in drug_classes_to_consider: #+ ["NSAID_PPI"]:
    drug_counts = {}
    for site in bleedintg_sites:
        count = site_major_df[(site_major_df[drug]==1) & (site_major_df[site]==1)]["patid"].nunique()
        drug_counts[site] = count
    bleeding_types_and_drug_classes_counts[drug] = drug_counts

bleeding_types_and_drug_classes_counts_df = pd.DataFrame(bleeding_types_and_drug_classes_counts).T
bleeding_types_and_drug_classes_counts_df["Total Major Bleeding Events"] = bleeding_types_and_drug_classes_counts_df.sum(axis=1)
bleeding_types_and_drug_classes_counts_df


In [ ]:
##remove antifungal and urogenital bleeding type from site-specific analysis due to low event counts
sites = {
    "GI":           "gi_bleeding",
    "Intracranial": "intracranial_bleeding",
    "Respiratory":  "respiratory_bleeding",
    "Urogenital":   "urogenital_bleeding",
    "Other":        "other_bleeding",
}

_base_cols = (["patid", "start_date", "end_date"]
              + drug_classes_to_consider
              + binary_comorbidity_cols + continuous_comorbidity_cols)

# Two fits per site: one with PPI_baseline, one without.
site_specs = {}
for site_name, event_col in sites.items():
    site_specs[f"{site_name}|no_ppi"] = (
        covariates_no_ppi, _base_cols + [event_col], event_col)
    site_specs[f"{site_name}|with_ppi"] = (
        covariates_with_ppi, _base_cols + [event_col, "PPI_baseline"], event_col)

# The fit moved below: it now runs once per imputed dataset and is pooled.
# site_specs is still built here and reused by the complete-case sensitivity
# analysis further down.


In [ ]:
# ---- MICE, one run per bleeding site ---------------------------------------
# The imputation model must condition on the outcome it will be used with, and
# each site is a different (cause-specific) outcome. So the site models get one
# imputation set each rather than a single shared one keyed on any-serious
# bleeding. With 5 sites and m=MI_CONFIG["m"] this is the slowest cell in the
# notebook; MICE itself is parallelised across imputed datasets.

site_imputations = {}
site_imp_diags = {}
for _site_name, _event_col in sites.items():
    _frame = build_imputation_frame(
        site_major_df, comorbidities,
        binary_comorbidity_cols, continuous_comorbidity_cols,
        drug_terms=drug_classes_to_consider, extra_binary=("PPI_baseline",),
        event_col=_event_col)
    _imps, _ = run_mice(_frame, IMPUTE_COLS, label=f"{_site_name} bleeding", **MI_CONFIG)
    site_imputations[_site_name] = _imps
    site_imp_diags[_site_name] = imputation_diagnostics(_frame, _imps, IMPUTE_COLS)
    site_imp_diags[_site_name].insert(0, "site", _site_name)

site_imp_diag = pd.concat(site_imp_diags.values(), ignore_index=True)
site_imp_diag


In [ ]:
# ---- Site-specific models, pooled by Rubin's rules --------------------------
# Each site is pooled over its own imputation set, so the fits are run one site
# at a time rather than as a single flat pool.
site_fits = {}
for _site_name, _event_col in sites.items():
    _specs = {
        f"{_site_name}|no_ppi": (covariates_no_ppi, _base_cols + [_event_col], _event_col),
        f"{_site_name}|with_ppi": (covariates_with_ppi,
                                   _base_cols + [_event_col, "PPI_baseline"], _event_col),
    }
    site_fits.update(fit_models_mi(site_major_df, site_imputations[_site_name],
                                   _specs, label=f"Site {_site_name}"))


In [ ]:
site_summaries = {}
for site_name in sites:
    rows = []
    for drug in drug_classes_to_consider:
        summary = model_for(site_fits, drug, prefix=f"{site_name}|")["summary"]
        if drug not in summary.index:
            continue
        r = summary.loc[drug]
        rows.append({
            "drug_class": drug,
            "PPI adjusted": ppi_flag(drug),
            "HR": round(r["exp(coef)"], 2),
            "CI Lower": round(r["exp(coef) lower 95%"], 2),
            "CI Upper": round(r["exp(coef) upper 95%"], 2),
            "p": round(r["p"], 4),
        })
    site_summaries[site_name] = (
        pd.DataFrame(rows).sort_values("HR", ascending=False).reset_index(drop=True))


site_summaries_df = pd.DataFrame()
for site_name, summary in site_summaries.items():
    summary = summary.copy()
    summary["site"] = site_name
    site_summaries_df = pd.concat([site_summaries_df, summary], ignore_index=True)

site_summaries_df.sort_values(by=["drug_class", "site"], ascending=[True, False], inplace=True)
site_summaries_df


In [ ]:
site_summaries_df.to_excel(os.path.join(SUPP_DIR, "site_specific_results.xlsx"), index=False)

In [ ]:
site_summaries_df =  pd.read_excel(os.path.join(SUPP_DIR, "site_specific_results.xlsx"))
site_summaries_df = site_summaries_df.drop(columns=["PPI adjusted"], errors="ignore")

In [ ]:
site_summaries_df

In [ ]:
# Plotting site-specific HRs
fig, ax = plt.subplots(figsize=(15, 10))

drug_classes = site_summaries_df["drug_class"].unique()
n_drugs = len(drug_classes)
y_pos = np.arange(n_drugs)
site_list = list(sites.keys())
n_sites = len(site_list)

site_colors = {
    site: color for site, color in zip(
        site_list,
        ['#B30000', '#005BBB', '#4dac26', '#8B00FF', '#FF8C00']
    )
}

for i, site in enumerate(site_list):
    site_data = site_summaries_df[site_summaries_df["site"] == site]
    site_data = site_data.set_index("drug_class").reindex(drug_classes).reset_index()
    offset = (i - (n_sites - 1) / 2) * 0.15
    color = site_colors[site]

    ax.errorbar(
        site_data['HR'], y_pos + offset,
        xerr=[site_data['HR'] - site_data['CI Lower'],
              site_data['CI Upper'] - site_data['HR']],
        fmt='o', label=site, color=color, ecolor=color,
        elinewidth=1.5, capsize=3, markersize=5
    )

# Separator lines between drug classes
for i in range(n_drugs - 1):
    ax.axhline(y=i + 0.5, color='grey', linestyle=':', linewidth=0.8, alpha=0.6)

# Alternating row shading
for i in range(n_drugs):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color='grey', alpha=0.05)

ax.axvline(x=1, color='black', linestyle='--', linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(drug_classes)
ax.invert_yaxis()

ax.set_xscale('log')
ax.set_xlim(0.3, 4)
ax.set_xticks([0.3, 1, 2, 3,4])
ax.set_xticklabels(['0.3', '1', '2', '3','4'])

ax.set_xlabel('Hazard Ratio')
ax.set_ylabel('Drug Class', fontsize=14)

transform = ax.get_yaxis_transform()

# ── Column positions: right side ──────────────────────────────────────────────
col_hr   = 1.06
col_pval = 1.15

# Column headers (once, above top row)
ax.text(col_hr,   -0.7, 'HR (95% CI)', transform=transform,
        ha='center', va='center', fontsize=9, fontweight='bold')
ax.text(col_pval, -0.7, 'p-value',     transform=transform,
        ha='center', va='center', fontsize=9, fontweight='bold')

# Per-site, per-drug annotations
for i, site in enumerate(site_list):
    site_data = site_summaries_df[site_summaries_df["site"] == site]
    site_data = site_data.set_index("drug_class").reindex(drug_classes).reset_index()
    color = site_colors[site]

    for j, drug in enumerate(drug_classes):
        subset = site_data[site_data["drug_class"] == drug]
        if subset.empty:
            continue
        row = subset.iloc[0]
        if pd.isna(row["HR"]):
            continue

        offset = (i - (n_sites - 1) / 2) * 0.15
        y = y_pos[j] + offset

        hr_str = f"{row['HR']:.2f} ({row['CI Lower']:.2f}–{row['CI Upper']:.2f})"
        p_str  = f"{row['p']:.3f}" if row['p'] >= 0.001 else "<0.001"

        ax.text(col_hr,   y, hr_str, transform=transform,
                ha='center', va='center', fontsize=8, color=color)
        ax.text(col_pval, y, p_str,  transform=transform,
                ha='center', va='center', fontsize=8, color=color)

ax.legend(loc='upper left', bbox_to_anchor=(1.2, 1))

OUTPUT_PNG = os.path.join(SUPP_FIGURES_DIR, "site_specific_HR_plot.png")
fig.savefig(OUTPUT_PNG, dpi=200, bbox_inches="tight", facecolor="white", transparent=False)
plt.tight_layout()
plt.show()

In [ ]:
doac_names = ["Apixaban", "Dabigatran", "Edoxaban", "Rivaroxaban"]

doac_codes = prod_dict[prod_dict["DrugSubstanceName"].str.contains("|".join(doac_names), case=False, na=False)]
apixaban_codes = doac_codes[doac_codes["DrugSubstanceName"]=="Apixaban"]["ProdCodeId"].tolist()
rivaroxaban_codes = doac_codes[doac_codes["DrugSubstanceName"]=="Rivaroxaban"]["ProdCodeId"].tolist()
dabigatran_codes = doac_codes[doac_codes["DrugSubstanceName"]=="Dabigatran etexilate mesilate"]["ProdCodeId"].tolist()
edoxaban_codes = doac_codes[doac_codes["DrugSubstanceName"]=="Edoxaban tosilate"]["ProdCodeId"].tolist()

doac_codes_list = apixaban_codes + rivaroxaban_codes + dabigatran_codes + edoxaban_codes

In [ ]:
zip_path = DRUG_ISSUE_ZIP

def process_file(file_name):
    with zipfile.ZipFile(zip_path) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep='\t', dtype=str)

    # Filtering
    df = df[(df['prodcodeid'].isin(doac_codes_list)) & (df['patid'].isin(serious_doac_exposure["patid"].tolist()))]
    df["study_start_date"] = df["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"])
    df["study_end_date"] = df["patid"].map(serious_doac_exposure.set_index("patid")["study_end_date"])
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors="coerce")
    df = df[(df["issuedate"] == df["study_start_date"])]
    return df


if __name__ == "__main__":
    with zipfile.ZipFile(zip_path) as z:
        txt_files = [f for f in z.namelist() if f.endswith('.txt')]

    # Run in parallel
    with Pool(60) as pool:
        all_results = pool.map(process_file, txt_files)

    # Combine results
    doac_presc_df = pd.concat(all_results, ignore_index=True)


In [ ]:
apixaban_pats = doac_presc_df[doac_presc_df["prodcodeid"].isin(apixaban_codes)]["patid"].unique()
rivaroxaban_pats = doac_presc_df[doac_presc_df["prodcodeid"].isin(rivaroxaban_codes)]["patid"].unique()
dabigatran_pats = doac_presc_df[doac_presc_df["prodcodeid"].isin(dabigatran_codes)]["patid"].unique()
edoxaban_pats = doac_presc_df[doac_presc_df["prodcodeid"].isin(edoxaban_codes)]["patid"].unique()

In [ ]:
apixaban_pats = set(apixaban_pats)
rivaroxaban_pats = set(rivaroxaban_pats)
dabigatran_pats = set(dabigatran_pats)
edoxaban_pats = set(edoxaban_pats)

major_df["apixaban"] = major_df["patid"].isin(apixaban_pats).astype(int)
major_df["rivaroxaban"] = major_df["patid"].isin(rivaroxaban_pats).astype(int)
major_df["dabigatran"] = major_df["patid"].isin(dabigatran_pats).astype(int)
major_df["edoxaban"] = major_df["patid"].isin(edoxaban_pats).astype(int)

In [ ]:
#drug_classes_to_consider = ["NSAID", "Antiplatelet", "SSRI", "CCB", "SNRI", "Antibiotic", "Antifungal"]

drugs_formula = " + ".join(drug_classes_to_consider)

doac_dummies = ["rivaroxaban", "edoxaban", "dabigatran"]   # apixaban is the reference

doac_formula = " + ".join(doac_dummies)

interaction_formula = " + ".join([
    f"{drug}:{doac}"
    for drug in drug_classes_to_consider
    for doac in doac_dummies
])

doac_covariates_no_ppi = (drugs_formula + " + " + doac_formula + " + "
                          + interaction_formula + " + " + comorbidities_covariates)
doac_covariates_with_ppi = doac_covariates_no_ppi + " + PPI_baseline"

_doac_cols = (["patid", "start_date", "end_date", "event_flag"]
              + drug_classes_to_consider + doac_dummies
              + binary_comorbidity_cols + continuous_comorbidity_cols)

doac_specs = {
    "no_ppi":   (doac_covariates_no_ppi, _doac_cols, "event_flag"),
    "with_ppi": (doac_covariates_with_ppi, _doac_cols + ["PPI_baseline"], "event_flag"),
}

covariates = doac_covariates_no_ppi


In [ ]:
# ---- MICE for the DOAC-type interaction analysis ----------------------------
# One imputation set: the outcome here is any serious bleeding (event_flag). The
# DOAC dummies join the imputation model as ever-exposed flags so it stays
# congenial with the interaction model.
doac_imp_frame = build_imputation_frame(
    major_df, comorbidities,
    binary_comorbidity_cols, continuous_comorbidity_cols,
    drug_terms=drug_classes_to_consider + doac_dummies,
    extra_binary=("PPI_baseline",), event_col="event_flag")

print(doac_imp_frame[["fu_years", "event", "na_cumhaz"]].describe().round(3))

doac_imputations, doac_mice_seeds = run_mice(
    doac_imp_frame, IMPUTE_COLS, label="DOAC type", **MI_CONFIG)

doac_imp_diag = imputation_diagnostics(doac_imp_frame, doac_imputations, IMPUTE_COLS)
doac_imp_diag


In [ ]:
# ---- DOAC-type models, pooled by multivariate Rubin's rules -----------------
# rubin_pool_mv returns a pooled params vector AND a pooled covariance matrix, so
# the combined main-effect-plus-interaction contrast computed further down uses
# the correct pooled covariance rather than a within-imputation one.
doac_fits = fit_models_mi(major_df, doac_imputations, doac_specs, label="DOAC type")


In [ ]:
# Interaction terms, each taken from the model appropriate to its drug class.
rows = []
for drug in drug_classes_to_consider:
    summary = model_for(doac_fits, drug)["summary"]
    for doac in doac_dummies:
        term = f"{drug}:{doac}"
        if term not in summary.index:
            continue
        r = summary.loc[term]
        rows.append({
            "drug_class": term,
            "PPI adjusted": ppi_flag(drug),
            "HR": round(r["exp(coef)"], 2),
            "CI Lower": round(r["exp(coef) lower 95%"], 2),
            "CI Upper": round(r["exp(coef) upper 95%"], 2),
            "p": round(r["p"], 4),
        })

major_summary_ctv = (pd.DataFrame(rows)
                     .sort_values("HR", ascending=False)
                     .reset_index(drop=True))
major_summary_ctv


In [ ]:
major_summary_ctv.to_excel(os.path.join(SUPP_DIR, "doac_type_and_drug_class_results.xlsx"), index=False)

In [ ]:
major_summary_ctv = pd.read_excel(os.path.join(SUPP_DIR, "doac_type_and_drug_class_results.xlsx"))

In [ ]:
doac_dummies = ["rivaroxaban", "edoxaban", "dabigatran"]
all_doacs    = ["apixaban"] + doac_dummies

results = []

for drug in drug_classes_to_consider:
    fit = model_for(doac_fits, drug)          # PPI-adjusted fit for NSAID only
    params, vcov, summary = fit["params"], fit["vcov"], fit["summary"]

    for doac in all_doacs:

        if doac == "apixaban":
            log_hr = params[drug]
            var    = vcov.at[drug, drug]

        else:
            interaction_term = f"{drug}:{doac}"
            log_hr = params[drug] + params[interaction_term]
            var    = (vcov.at[drug, drug]
                    + vcov.at[interaction_term, interaction_term]
                    + 2 * vcov.at[drug, interaction_term])

        se = np.sqrt(var)

        p_interaction = None
        if doac != "apixaban":
            interaction_term = f"{drug}:{doac}"
            if interaction_term in summary.index:
                p_interaction = summary.loc[interaction_term, "p"]

        results.append({
            "drug_class":    drug,
            "doac":          doac,
            "PPI adjusted":  ppi_flag(drug),
            "HR":            round(np.exp(log_hr), 2),
            "CI Lower":      round(np.exp(log_hr - 1.96 * se), 2),
            "CI Upper":      round(np.exp(log_hr + 1.96 * se), 2),
            "p_interaction": round(p_interaction, 4) if p_interaction is not None else None
        })

results_df = pd.DataFrame(results)
results_df.sort_values(by=["drug_class", "doac"], inplace=True)
results_df


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

doac_colors = {
    "apixaban":    "#2166ac",
    "rivaroxaban": "#d6604d",
    "edoxaban":    "#4dac26",
    "dabigatran":  "#8B00FF"
}
doac_offsets = {
    "apixaban":    -0.22,
    "rivaroxaban": -0.07,
    "edoxaban":     0.07,
    "dabigatran":   0.22
}

drugs     = drug_classes_to_consider
n_drugs   = len(drugs)
y_pos     = np.arange(n_drugs)
all_doacs = ["apixaban", "rivaroxaban", "edoxaban", "dabigatran"]

fig, ax = plt.subplots(figsize=(15, 8))

for doac in all_doacs:
    subset   = results_df[results_df["doac"] == doac].set_index("drug_class")
    offset   = doac_offsets[doac]
    color    = doac_colors[doac]

    hrs      = [subset.loc[d, "HR"]       for d in drugs]
    ci_lower = [subset.loc[d, "CI Lower"] for d in drugs]
    ci_upper = [subset.loc[d, "CI Upper"] for d in drugs]

    ax.errorbar(
        hrs, y_pos + offset,
        xerr=[
            [h - l for h, l in zip(hrs, ci_lower)],
            [u - h for h, u in zip(hrs, ci_upper)]
        ],
        fmt="o", color=color, ecolor=color,
        elinewidth=1.5, capsize=3, markersize=5,
        label=doac.capitalize()
    )

# Separator lines between drug classes
for i in range(n_drugs - 1):
    ax.axhline(y=i + 0.5, color='grey', linestyle=':', linewidth=0.8, alpha=0.6)

# Alternating row shading
for i in range(n_drugs):
    if i % 2 == 0:
        ax.axhspan(i - 0.5, i + 0.5, color='grey', alpha=0.05)

# Reference line
ax.axvline(x=1, color="black", linestyle="--", linewidth=1)

# Y axis
ax.set_yticks(y_pos)
ax.set_yticklabels(drugs, fontsize=11)
ax.invert_yaxis()

# X axis
ax.set_xscale("log")
ax.set_xlim(0.1, 10)
ax.set_xticks([0.25, 0.5, 1, 2, 4, 8])
ax.set_xticklabels(["0.25", "0.5", "1", "2", "4", "8"])
ax.set_xlabel("Hazard Ratio (log scale)", fontsize=11)

transform = ax.get_yaxis_transform()

# ── Column positions (axes fraction) ─────────────────────────────────────────
col_hr   = 1.05
col_pval = 1.135

# Column headers
ax.text(col_hr,   -0.6, "HR (95% CI)", transform=transform,
        ha='center', va='center', fontsize=9, fontweight='bold')
ax.text(col_pval, -0.6, "p-value", transform=transform,
        ha='center', va='center', fontsize=9, fontweight='bold')

# Per-doac, per-drug annotations
for i, drug in enumerate(drugs):
    for doac in all_doacs:
        row    = results_df[(results_df["drug_class"] == drug) & (results_df["doac"] == doac)].iloc[0]
        offset = doac_offsets[doac]
        color  = doac_colors[doac]
        y      = y_pos[i] + offset

        hr_str = f"{row['HR']:.2f} ({row['CI Lower']:.2f}–{row['CI Upper']:.2f})"

        p_int  = row["p_interaction"]
        if doac == "apixaban":
            p_str = "ref"
        elif p_int is None or pd.isna(p_int):
            p_str = "—"
        else:
            p_str = f"{p_int:.3f}" if p_int >= 0.001 else "<0.001"

        ax.text(col_hr,   y, hr_str, transform=transform,
                ha='center', va='center', fontsize=8, color=color)
        ax.text(col_pval, y, p_str,  transform=transform,
                ha='center', va='center', fontsize=8, color=color)

# Significance note
#ax.text(0.11, n_drugs - 0.3, "* p-interaction < 0.05 vs apixaban",
#        fontsize=8, color="grey", style="italic")

ax.legend(title="DOAC", bbox_to_anchor=(1.18, 1), loc="upper left", fontsize=10)
ax.set_title("Bleeding Hazard Ratios by Drug Class and DOAC Type\n", fontsize=12, pad=12)

OUTPUT_PNG = os.path.join(SUPP_FIGURES_DIR, "bleeding_HR_by_DOAC_type.png")
fig.savefig(OUTPUT_PNG, dpi=200, bbox_inches="tight", facecolor="white", transparent=False)
plt.tight_layout()
plt.show()